# nb_preparar_entorno
| Campo | Valor |
|---|---|
| Proyecto | Andina Market · Reto técnico de Ingeniería de Datos |
| Capa | Ingesta · preparación (tarea 1 del job `job_andina_diario`) |
| Propósito | Crear los esquemas y Volumes que necesita la ingesta, en cualquier entorno |
| Parámetros | `catalogo` (andina_dev · andina_prd), `storage_account` (ADLS del entorno) |
| Idempotencia | Todo usa `IF NOT EXISTS`; los comentarios de esquema se reaplican con `COMMENT ON` |
| Autoría | Richard Huamán · con asistencia de IA (Claude) |
| Versión | 1.1 · 2026-10-20 · comentario de esquema también en esquemas ya existentes |

**Por qué una tarea y no recursos del bundle:** los objetos de dev se crearon a mano antes del bundle. Con `IF NOT EXISTS` la misma tarea los respeta en dev y los crea en prd, sin tener que vincularlos al bundle.

## 1. Parámetros

In [0]:
dbutils.widgets.text("catalogo", "andina_dev", "Catálogo (ambiente)")
dbutils.widgets.text("storage_account", "standinadevcus001", "Cuenta ADLS Gen2 del ambiente")
# Carpeta de landing dentro del contenedor: "landing" en dev y prd (cada uno con su storage);
# "stg/landing" en stg, que comparte el storage de dev y no puede solapar sus Volumes.
dbutils.widgets.text("carpeta_landing", "landing", "Carpeta de landing en el contenedor")

CATALOGO = dbutils.widgets.get("catalogo").strip()
STORAGE = dbutils.widgets.get("storage_account").strip()
CARPETA_LANDING = dbutils.widgets.get("carpeta_landing").strip().strip("/")
RAIZ_LANDING = f"abfss://lakehouse@{STORAGE}.dfs.core.windows.net/{CARPETA_LANDING}"
print(f"catálogo={CATALOGO} · landing={RAIZ_LANDING}")

## 2. Esquemas
- `landing`: archivos tal como llegan (Parquet del extractor, .jsonl del clickstream).
- `control`: metadatos operativos de la ingesta (objetos, watermarks, ejecuciones).
- `bronze`: tablas crudas, solo append.

In [0]:
esquemas = {
    "landing": "Archivos tal como llegan de las fuentes, antes de Bronze.",
    "control": "Metadatos operativos de la ingesta: objetos, watermarks y ejecuciones.",
    "bronze":  "Capa bronze: datos crudos tal como llegan de la fuente, append-only, con metadatos de ingesta",
}
for esquema, comentario in esquemas.items():
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{esquema} COMMENT '{comentario}'")
    # IF NOT EXISTS no toca un esquema existente: COMMENT ON corrige los creados sin comentario
    spark.sql(f"COMMENT ON SCHEMA {CATALOGO}.{esquema} IS '{comentario}'")
    print(f"  ✓ {CATALOGO}.{esquema}")

## 3. Volumes
- **Externos** en landing: los archivos viven en el ADLS del entorno (`landing/<fuente>`), independientes del workspace.
- **Administrado** para checkpoints: estado interno de Auto Loader; no se borra (garantiza exactly-once).

Requiere la ubicación externa del entorno (`loc_andina_dev` / `loc_andina_prd`) sobre `abfss://lakehouse@<storage>...`.

In [0]:
volumenes_externos = {
    "sqlserver":   "Parquet del extractor CDC de Azure SQL (una carpeta por tabla y fecha). Retención 90 días.",
    "clickstream": "Eventos de la app móvil en .jsonl (muestra del reto). Retención 90 días.",
}
for nombre, comentario in volumenes_externos.items():
    spark.sql(f"""CREATE EXTERNAL VOLUME IF NOT EXISTS {CATALOGO}.landing.{nombre}
                  LOCATION '{RAIZ_LANDING}/{nombre}' COMMENT '{comentario}'""")
    print(f"  ✓ {CATALOGO}.landing.{nombre} → {RAIZ_LANDING}/{nombre}")

spark.sql(f"""CREATE VOLUME IF NOT EXISTS {CATALOGO}.bronze.checkpoints
              COMMENT 'Checkpoints y schemaLocation de Auto Loader. No borrar: garantiza exactly-once.'""")
print(f"  ✓ {CATALOGO}.bronze.checkpoints (administrado)")

## 4. Verificación

In [0]:
display(spark.sql(f"""
    SELECT volume_schema, volume_name, volume_type, storage_location
    FROM {CATALOGO}.information_schema.volumes
    WHERE volume_schema IN ('landing', 'bronze')
    ORDER BY volume_schema, volume_name"""))